In [107]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import numpy as np
from langchain_core.documents import Document
import re
from rank_bm25 import BM25Okapi
import pickle
import sys, os
sys.path.append(os.path.abspath(".."))
from src.bm25 import load_bm25, bm25_search
from src.utils import tokenize, tokenize_corpus

In [108]:

CATEGORY = "Kindle_Store"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"

REVIEWS_FILE = RAW_DIR / "reviews_raw.parquet"
META_FILE = RAW_DIR / "meta_raw.parquet"
OUTPUT_FILE = PROCESSED_DIR / f"{CATEGORY}_merged.parquet"

In [109]:
c2 = duckdb.connect()

## EDA of Raw Review and Metadata Datasets

In [110]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,excellent writer reminds me of Clive Cussler,GRUMLEY is on par with Clive Cussler and his D...,[],B00LXRJICK,B00LXRJICK,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1427541413000,0,False
1,3.0,Alright book,The book Fade was not my favorite but was a go...,[],B073DFP8VC,B073DFP8VC,AHGTHCERTEZUXNBLJ5SWHK2CDLXA,1504226946142,0,True
2,5.0,Hats off to Fern Michaels for all her great ac...,I have been a fan of this author for many year...,[],B07QVH25KX,B07QVH25KX,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1644883955777,0,True
3,5.0,Great read,This book is more than just about a dog and a ...,[],B004Y1NWQU,B004Y1NWQU,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1363027885000,0,False
4,5.0,Add to legend f Arthur,Good twist on the ledgen of King<br />Arthur !...,[],B08M993CNC,B08M993CNC,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1637557512064,0,True


In [111]:
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Buy a Kindle,The Palace (Chateau Book 4),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.7,970,[A line was drawn in the sand.I made my choice...,[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Penelope Sky (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Romance]","{'Publisher': 'Hartwick Publishing (May 25, 20...",B08XPZPFY4,None
1,Buy a Kindle,Microsoft PowerPoint 2016 2013 2010 2007 Tips ...,[Print Replica] Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.3,35,"[Paperback versions are also available, includ...","[From the Author, Amelia Griggs is an Instruct...",0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Amelia Griggs (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Computers & Tech...","{'Publisher': None, 'Publication date': 'June ...",B07DH1LF1K,None
2,Buy a Kindle,Ill Wind (Anna Pigeon Mysteries Book 3),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.4,1628,"[Lately, visitors to Mesa Verde have been brin...","[From Publishers Weekly, Barr lands another su...",7.99,[{'large': 'https://m.media-amazon.com/images/...,[],Nevada Barr (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",{'Publisher': 'Berkley; Reissue edition (March...,B0022Q8CTQ,None
3,Buy a Kindle,30 Healthy Easy Quick Lentil Recipes (Brad Arm...,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.8,47,"[If improved health you are seeking, look no f...",[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Brad Armstrong (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Health, Fitness ...","{'Publisher': 'Brad Armstrong (March 10, 2013)...",B00BS56MLC,None
4,Buy a Kindle,The Road Home,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,475,"[In one of Jim Harrison’s greatest works, five...","[Review, ""A graceful novel...To read this book...",10.44,[{'large': 'https://m.media-amazon.com/images/...,[],Jim Harrison (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Literature & Fic...",{'Publisher': 'Atlantic Monthly Press (Decembe...,B00155EZRS,None


In [112]:
c2.execute(f"""
    COPY ( SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 275000)
    TO '{REVIEWS_FILE}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [113]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 50000)
      TO '{META_FILE}'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [114]:
reviews_df = c2.execute(f"SELECT * FROM read_parquet('{REVIEWS_FILE}')").df()
meta_df = c2.execute(f"SELECT * FROM read_parquet('{META_FILE}')").df()

In [115]:
reviews_df.describe(), reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 275000 entries, 0 to 274999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             275000 non-null  float64
 1   title              275000 non-null  str    
 2   text               275000 non-null  str    
 3   images             275000 non-null  object 
 4   asin               275000 non-null  str    
 5   parent_asin        275000 non-null  str    
 6   user_id            275000 non-null  str    
 7   timestamp          275000 non-null  int64  
 8   helpful_vote       275000 non-null  int64  
 9   verified_purchase  275000 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 147.7+ MB


(              rating     timestamp   helpful_vote
 count  275000.000000  2.750000e+05  275000.000000
 mean        4.384175  1.515352e+12       0.846156
 std         0.959101  8.846120e+10       6.699364
 min         1.000000  1.127106e+12       0.000000
 25%         4.000000  1.443482e+12       0.000000
 50%         5.000000  1.514061e+12       0.000000
 75%         5.000000  1.588790e+12       1.000000
 max         5.000000  1.679317e+12    1253.000000,
 None)

In [116]:

meta_df.describe(), meta_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    49998 non-null  str    
 1   title            50000 non-null  str    
 2   subtitle         49084 non-null  str    
 3   author           42206 non-null  object 
 4   average_rating   50000 non-null  float64
 5   rating_number    50000 non-null  int64  
 6   features         50000 non-null  object 
 7   description      50000 non-null  object 
 8   price            47257 non-null  float64
 9   images           50000 non-null  object 
 10  videos           50000 non-null  object 
 11  store            50000 non-null  str    
 12  categories       50000 non-null  object 
 13  details          50000 non-null  object 
 14  parent_asin      50000 non-null  str    
 15  bought_together  0 non-null      object 
dtypes: float64(2), int64(1), object(8), str(5)
memory usage: 13.1+ MB


(       average_rating  rating_number         price
 count    50000.000000   50000.000000  47257.000000
 mean         4.307856     424.006680      3.857522
 std          0.513857    2518.897923      8.333157
 min          1.000000       1.000000      0.000000
 25%          4.100000      12.000000      0.000000
 50%          4.400000      45.000000      0.990000
 75%          4.600000     198.000000      4.990000
 max          5.000000  138549.000000    982.990000,
 None)

## EDA of the Merged  Dataset

In [117]:
# Select only the required columns and perform an INNER JOIN to merge reviews and meta.
# Note: A LEFT JOIN is not used to prevent introducing many missing values.

c2.execute("""
    COPY (
         SELECT r.parent_asin, r.rating, r.title AS review_title, r.text AS review_text, r.verified_purchase,
            m.title AS product_title, m.average_rating, m.main_category, m.description, m.features
        FROM read_parquet('../data/raw/reviews_raw.parquet') r
        INNER JOIN read_parquet('../data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO '../data/processed/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [118]:
merged_df = c2.execute(f"SELECT * FROM read_parquet('../data/processed/merged.parquet')").df()

In [119]:
merged_df.head(5)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B003I1WY2A,4.0,Hid the attacker well. Ending was totally unbe...,Hid the attacker well. Ending was totally unb...,False,Water for Elephants: A Novel,4.5,Buy a Kindle,"[About the Author, SARA GRUEN was born in Vanc...","[Over 10,000,000 copies in print worldwide #1,..."
1,B00DQUKZMY,4.0,"Well written, quick read.","This is a fast moving, light reading story. Th...",False,To Honor You Call Us (Man of War Book 1),4.4,Buy a Kindle,"[Review, “Horatio Hornblower in Space!”, —The ...",[The Terran Union is engaged in a vast interst...
2,B0071NPT4Q,4.0,READ MOST TALKATIVE BEFORE TEH DIARIES,The stories are great from coming out to Susan...,True,Most Talkative: Stories from the Front Lines o...,4.4,Buy a Kindle,"[About the Author, Andy Cohen, is Bravo's exec...","[The man behind the, Real Housewives, writes a..."
3,B08DQLQDBF,5.0,Delightful read,First book I have read by Karen McQuestion. I ...,False,The Moonlight Child,4.4,Buy a Kindle,"[From the Author, PRAISE FOR THE BOOKS OF KARE...",[“I cannot recommend this book highly enough! ...
4,B008AV8J6I,5.0,Secrets,This book is so interesting I could not put it...,False,Secrets (The Michelli Family Series Book #1): ...,4.4,Buy a Kindle,"[About the Author, Kristen Heitzmann is a full...",[Lance Michelli is on a quest to discover the ...


In [120]:
merged_df.info(), merged_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10450 entries, 0 to 10449
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   parent_asin        10450 non-null  str    
 1   rating             10450 non-null  float64
 2   review_title       10450 non-null  str    
 3   review_text        10450 non-null  str    
 4   verified_purchase  10450 non-null  bool   
 5   product_title      10450 non-null  str    
 6   average_rating     10450 non-null  float64
 7   main_category      10450 non-null  str    
 8   description        10450 non-null  object 
 9   features           10450 non-null  object 
dtypes: bool(1), float64(2), object(2), str(5)
memory usage: 5.8+ MB


(None,
              rating  average_rating
 count  10450.000000    10450.000000
 mean       4.378852        4.351158
 std        0.951455        0.288523
 min        1.000000        1.000000
 25%        4.000000        4.200000
 50%        5.000000        4.400000
 75%        5.000000        4.500000
 max        5.000000        5.000000)

In [121]:
merged_df.isna().mean().sort_values(ascending=False)

parent_asin          0.0
rating               0.0
review_title         0.0
review_text          0.0
verified_purchase    0.0
product_title        0.0
average_rating       0.0
main_category        0.0
description          0.0
features             0.0
dtype: float64

In [122]:

# Function to convert a list of strings into a single string

def to_text(x):
    if x is pd.NA:
        return ""
    if isinstance(x, np.ndarray):
        return " ".join(map(str, x))
    return str(x)

merged_df["features"] = merged_df["features"].apply(to_text)
merged_df["description"] = merged_df["description"].apply(to_text)

In [123]:
merged_df.head(3)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B003I1WY2A,4.0,Hid the attacker well. Ending was totally unbe...,Hid the attacker well. Ending was totally unb...,False,Water for Elephants: A Novel,4.5,Buy a Kindle,About the Author SARA GRUEN was born in Vancou...,"Over 10,000,000 copies in print worldwide #1 N..."
1,B00DQUKZMY,4.0,"Well written, quick read.","This is a fast moving, light reading story. Th...",False,To Honor You Call Us (Man of War Book 1),4.4,Buy a Kindle,Review “Horatio Hornblower in Space!” —The Ame...,The Terran Union is engaged in a vast interste...
2,B0071NPT4Q,4.0,READ MOST TALKATIVE BEFORE TEH DIARIES,The stories are great from coming out to Susan...,True,Most Talkative: Stories from the Front Lines o...,4.4,Buy a Kindle,About the Author Andy Cohen is Bravo's executi...,The man behind the Real Housewives writes abou...


In [124]:
# From the selected columns, investigate which ones are meaningful based on their number of unique values
merged_df.nunique()

parent_asin           5400
rating                   5
review_title          8310
review_text          10260
verified_purchase        2
product_title         5397
average_rating          32
main_category            2
description           3105
features              5335
dtype: int64

In [125]:
# Create the document column by combining the fields that provide the most context
merged_df["document"] = (
    "Product: " + merged_df["product_title"].fillna("") + ". "
    + "Description: " + merged_df["description"].fillna("") + ". "
    + "Features: " + merged_df["features"].fillna("") + ". "
    + "Review Title: " + merged_df["review_title"].fillna("") + ". "
    + "Review: " + merged_df["review_text"].fillna("")
)

In [126]:
merged_df['document'][1]

'Product: To Honor You Call Us (Man of War Book 1). Description: Review “Horatio Hornblower in Space!” —The American Catholic Review "Horatio Hornblower in Space!" —The American Catholic About the Author H. Paul Honsinger is a retired attorney. Born and raised in Lake Charles, Louisiana, he is a graduate of Lake Charles High School, The University of Michigan in Ann Arbor, and Louisiana State University Law School in Baton Rouge. Honsinger currently lives in Lake Havasu City, Arizona with his beloved wife, Kathleen, and his daughter and stepson, as well as a 185-pound English Mastiff and two highly eccentric cats. Paul’s hobbies include astronomy, military history, and the history of the Apollo Program. Review “Horatio Hornblower in Space!” ―The American Catholic --This text refers to an out of print or unavailable edition of this title. Read more. Features: The Terran Union is engaged in a vast interstellar war against the Krag, ruthless aliens intent on exterminating humankind. In 23



### Text preprocessing decisions

To prepare the data for retrieval, we applied the following preprocessing steps to preserve useful context while making the text easier to search.

- We merged reviews with metadata using an INNER join on parent_asin so that every document contains both review text and product information.

- We kept informative text fields such as product_title, review_title, review_text, description, and features.

- We removed non-meaningful or non-text fields such as parent_asin, rating, verified_purchase, and main_category from the final document text.

- The description and features columns originally stored array values, so we converted them into single strings, and missing values were replaced with empty strings.

- We created a column named document which combines product metadata and review text using the following columns: Product, Description, Features, Review Title, and Review.

The preprocessing implemented is initial, and more retrieval-specific preprocessing such as lowercasing, tokenization, punctuation removal, and optional stopword removal will be applied in the upcoming stages.

### Preparing data for retrieval (LangChain format)

For retrieval stages (BM25 and semantic search), we convert each row into LangChain Document objects. This separates the searchable text `page_content' from the metadata.

In [127]:
# Convert each row into LangChain Document format.
# page_content stores searchable text, and metadata keeps identifiers and attributes.

langchain_docs = []

for _, row in merged_df.iterrows():
    doc = Document(
        page_content=row["document"],
        metadata={
            "parent_asin": row["parent_asin"],
            "product_title": row["product_title"],
            "rating": row["rating"],
            "verified_purchase": row["verified_purchase"],
            "average_rating": row["average_rating"],
            "main_category": row["main_category"],
        }
    )
    langchain_docs.append(doc)

In [128]:
print(langchain_docs[0].page_content)
print(langchain_docs[0].metadata)

Product: Water for Elephants: A Novel. Description: About the Author SARA GRUEN was born in Vancouver, lived in London, Ontario, andwent to university in Ottawa. She now lives with her husband and three childrenin a conservation community outside of Chicago. Her fiction debut, RidingLessons , was an international bestseller. Water for Elephants is herthird novel. Visit her website at www.saragruen.com. --This text refers to an out of print or unavailable edition of this title.. Features: Over 10,000,000 copies in print worldwide #1 New York Times Bestseller A Los Angeles Times Bestseller A Wall Street Journal Bestseller A Newsday Favorite Book of 2006 A USA Today Bestseller A Major Motion Picture starring Reese Witherspoon, Robert Pattinson, and Christoph Waltz Jacob Janowski’s luck had run out--orphaned and penniless, he had no direction until he landed on a rickety train that was home to the Benzini Brothers Most Spectacular Show on Earth. A veterinary student just shy of a degree, h

## BM25 Preprocessing

To prepare the text for BM25 , we applied the following tokenization transofrmations:
- Converted text to lowercase
- Removed punctuation
- Split text by whitespace

Note: These steps are implemented in src/utils.py.

In [129]:
# Create the corpus and tokenize it in preparation for BM25
corpus = merged_df["document"].tolist()
tokenized_corpus = tokenize_corpus(corpus)

In [130]:
tokenized_corpus[1][:5]

['product', 'to', 'honor', 'you', 'call']

## BM25 Index Construction

Constructed a BM25 index using the tokenized corpus from above.  As learnt during class, BM25 is a keyword-based retrieval method that ranks documents based on term frequency and inverse document frequency.

The index is saved to disk to avoid recomputing it each time.

In [131]:
# Build the BM25 index from the tokenized corpus
bm25_model = BM25Okapi(tokenized_corpus)

In [132]:

Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Save the tokenized corpus and BM25 index so they can be reused later without rebuilding
with open("../data/processed/tokenized_corpus.pkl", "wb") as f:
    pickle.dump(tokenized_corpus, f)


with open("../data/processed/bm25.pkl", "wb") as f:
    pickle.dump(bm25_model, f)

## BM25 Retrieval Example

The examples below show the top-ranked documents for the query.  
As expected, BM25 performs well for keyword-based queries where exact term matches are important.

In [133]:
# Retrieve BM25
bm25_model = load_bm25()

# Test with a query
query = "weight loss book"
results = bm25_search(query, bm25_model, merged_df, top_k=5)
results

,product_title,review_title,review_text,bm25_score,rating
5797,Lose Weight Fast: Over 50 Incredible Weight Lo...,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,16.831494,4.0
6622,HYPOTHYROIDISM DIET ~ The secrets to your thyr...,Delicious Lentil Recipe,If you are struggling to make any progress in ...,16.520224,5.0
9704,Nutritional Healing: Special Nutrition with th...,What is good for you to help heal to body,This is a very informative book. I was chosen...,15.633907,5.0
873,Fat Fast Cookbook: 50 Easy Recipes to Jump Sta...,Pros and cons,"Although I've been on Atkins for 5 weeks, I ha...",15.556894,4.0
8097,Yoga For Beginners: An Easy Yoga Guide To Reli...,For beginners?,"The text may be for beginners, encouraging yet...",15.508411,1.0


In [134]:
query = "data science "
bm25_ex2= bm25_search(query, bm25_model, merged_df, top_k=5)
bm25_ex2

,product_title,review_title,review_text,bm25_score,rating
9403,Learning JavaScript Data Structures and Algori...,but 85% of the book is a great resource. Factu...,"There were some errors in the book, but 85% of...",10.511885,3.0
4846,Twisted Spaces: 1 / Destination Mars,Twisted Spaces; Book 1,I've now read Herr Abel's Books 1 and 2. I'm o...,10.104474,3.0
4736,The Very First Light: The True Inside Story of...,Five Stars,Enjoyed hearing Dr. Boslough speak about his b...,9.670423,5.0
9405,Mastering JavaScript Object-Oriented Programming,Five Stars,A great read for getting into the OO sides of ...,9.320969,5.0
143,Head First JavaScript Programming: A Brain-Fri...,Five Stars,Very helpful book. I have taken several online...,9.283263,5.0


## BM25 Retrieval Results (10 Queries)

In [135]:
queries = [
    "weight loss book",
    "healthy eating guide",
    "beginner cookbook",

    "book for better habits",
    "book to relax before bed",
    "books about animals for kids",

    "best books for relaxing before sleep",
    "stories about friendship and growth",
    "good books for beginners learning investing",
    "novels for long flights"
]

In [136]:
bm25_outputs = {}

for q in queries:
    bm25_outputs[q] = bm25_search(q, bm25_model, merged_df, top_k=5)

In [137]:
for q in queries:
    
    print("=" * 50)
    print(f"Query: {q}")
    display(bm25_outputs[q])

Query: weight loss book


,product_title,review_title,review_text,bm25_score,rating
5797,Lose Weight Fast: Over 50 Incredible Weight Lo...,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,16.831494,4.0
6622,HYPOTHYROIDISM DIET ~ The secrets to your thyr...,Delicious Lentil Recipe,If you are struggling to make any progress in ...,16.520224,5.0
9704,Nutritional Healing: Special Nutrition with th...,What is good for you to help heal to body,This is a very informative book. I was chosen...,15.633907,5.0
873,Fat Fast Cookbook: 50 Easy Recipes to Jump Sta...,Pros and cons,"Although I've been on Atkins for 5 weeks, I ha...",15.556894,4.0
8097,Yoga For Beginners: An Easy Yoga Guide To Reli...,For beginners?,"The text may be for beginners, encouraging yet...",15.508411,1.0


Query: healthy eating guide


,product_title,review_title,review_text,bm25_score,rating
1588,Fast And Easy Cabbage Recipes: An Guide To An ...,healthy eating,oh yea i forgot i had this book... i need to d...,19.419065,4.0
8597,Mommy Muscles: A practical guide for building ...,I love her writing,I love her writing...I feel as if she is right...,16.777673,5.0
8,Healthy Family Cookbook: 100 Fast and Easy Rec...,"Easy, Simple and a wide selection!!",I have a odd love for cookbooks but rarely mak...,16.309771,5.0
6626,Blood Pressure: Blood Pressure Solution: The S...,Well Organized and Very Helpful,High blood pressure is common in my family. A ...,15.931009,5.0
1700,Everything Vegetarian: The Complete Cookbook f...,I have been a fan of Wendy's Vegetarian recipe...,"I received this book for free, actually I jump...",14.983249,5.0


Query: beginner cookbook


...

,product_title,review_title,review_text,bm25_score,rating
5800,A Widow Redefined,Touching !,i got this book from amazon as a freebie (or m...,18.571952,5.0
1695,A Widow Redefined,"Friendship born of grief,,,",A little mystery a lot of dealing with emotion...,18.338691,4.0
8839,I Can't Reach It!: A Growth Mindset Book To Pr...,Believe in yourself and use your imagination t...,I love the colorful illustrations in this seri...,18.204033,5.0
4077,Riversnow (River Valley Book 4),Riversnow,This is a beautiful story and so enjoyable. A...,18.122821,5.0
8329,Riversnow (River Valley Book 4),Great read.,I’m joining the other 91% and going for the 5 ...,17.470594,5.0


Query: good books for beginners learning investing


,product_title,review_title,review_text,bm25_score,rating
41,Taming the Tongue: The Power of Spoken Words,Great little book,I enjoyed this book very much. it is full of g...,32.921885,5.0
2041,Taming the Tongue: The Power of Spoken Words,a bible study and a cheap self-help book collided,This book had a lot of scriptures at the begin...,32.807765,2.0
8200,Taming the Tongue: The Power of Spoken Words,Baby Steps,This is a decent book for the young (physicall...,32.579318,3.0
44,NFTs for Beginners: Making Money with Non-Fung...,Easy overview,I liked the fact that it was an easy read in l...,23.675054,5.0
10369,Private Label: 7 Steps to Earning 1K to 5K per...,Nothing Really New,"It is a short read, some good information alth...",21.989952,3.0


Query: novels for long flights


,product_title,review_title,review_text,bm25_score,rating
3284,Sawyer Jackson and the Long Land,A very different view of realities and possibi...,Most enjoyable story of coming of age and inhe...,12.484750,5.0
614,Scrivener's Moon (Fever Crumb Triology Book 3),Another amazing book in the series,Loved this book. The writing style is wonderf...,12.011178,5.0
7273,The Christmas Gamble: A Billionaire Christmas ...,Kayla and Dean❤,Newly single Kayla is heading home to see her ...,11.742221,4.0
1022,Letters from Amelia,Four Stars,dragged a bit in parts,11.615034,4.0
9737,Savage Song: A Special Edition Destroyer Novel...,Politically Incorrect Laughs,I've been reading the Destroyer series for dec...,11.491768,4.0


## Semantic retrieval (dense)

This block adds **dense (embedding-based) retrieval** alongside BM25. Documents are embedded with the same `document` text as sparse retrieval, indexed with FAISS, then queried through `src.semantic` for a fair comparison on the same evaluation queries.


## Semantic index construction (Step 1)

We build a **dense retrieval** index so queries can be matched by meaning, not only overlapping keywords (unlike BM25).

### Inputs
- **`merged_df["document"]`**: the same concatenated review/product text used for BM25, so sparse and dense retrieval stay aligned on the corpus.

### Method
- **Encoder:** `sentence-transformers/all-MiniLM-L6-v2` maps each document to a fixed-size vector.
- **Normalization:** Embeddings are L2-normalized so **cosine similarity** equals the **inner product** between query and document vectors.
- **Index:** We use FAISS **`IndexFlatIP`** (exact inner-product search) over all document vectors. This is exact but simple; no training step is required for this index type.

### Outputs (saved under `data/processed/`)
- **`semantic_faiss.index`**: the FAISS index containing all document embeddings.
- **`semantic_meta.json`**: metadata (e.g. model name, number of documents, embedding dimension) so retrieval code can load the **same** model at query time.

### Notes
- The first run may **download** the SentenceTransformer weights from Hugging Face.
- If **`merged.parquet`** or the **`document`** text changes, this cell must be **re-run** to rebuild the index (same idea as rebuilding BM25).

### What the next code cell does (build and save)

- **Imports:** `Path` / `json` for paths and metadata; `faiss` for the vector index; `numpy` for float32 arrays; `SentenceTransformer` loads the bi-encoder that maps text to vectors.
- **`MODEL_NAME`:** must stay the same for **indexing and querying** (also recorded in `semantic_meta.json`).
- **`corpus`:** one string per row from `merged_df["document"]` — identical text to BM25’s corpus, so row `i` in the index matches `merged_df.iloc[i]`.
- **`model.encode(..., normalize_embeddings=True)`:** L2-normalizes vectors so **inner product** in FAISS equals **cosine similarity** between query and document.
- **`IndexFlatIP` + `add`:** exact nearest-neighbor search in embedding space (no clustering / compression).
- **`write_index` + `json.dump`:** persist the index and minimal metadata (`model_name`, `n_docs`, `dim`) for `load_semantic()`.


In [138]:
import json
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

PROCESSED = Path("../data/processed")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # same for index + queries

model = SentenceTransformer(MODEL_NAME)
corpus = merged_df["document"].astype(str).tolist()

doc_embeddings = model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2-normalize for cosine via inner product
)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))

faiss.write_index(index, str(PROCESSED / "semantic_faiss.index"))
with open(PROCESSED / "semantic_meta.json", "w") as f:
    json.dump({"model_name": MODEL_NAME, "n_docs": len(corpus), "dim": dim}, f)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6495.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 164/164 [00:28<00:00,  5.66it/s]


### Quick sanity check on the in-memory index

The next cell encodes **one** query with the **same** normalization as the corpus, runs `index.search` for the top **3** neighbors, and prints **row indices** and **scores**. You should see plausible neighbors for `"weight loss book"` and scores typically in a cosine-like range for normalized vectors. This is optional smoke testing before relying on the saved files.


In [139]:
q = model.encode(["weight loss book"], convert_to_numpy=True, normalize_embeddings=True)
scores, idx = index.search(q.astype(np.float32), 3)
print(idx, scores)

[[9979 5797 6639]] [[0.54520226 0.5357093  0.5351836 ]]


### Retrieval via `src.semantic` (Step 3)

- **`load_semantic()`:** reads `semantic_meta.json`, instantiates the **same** `SentenceTransformer` named there, and loads `semantic_faiss.index` from disk. If the index was built with a different model or corpus order, reload or rebuild the index.
- **`semantic_search(...)`:** embeds the query string, searches FAISS, maps hit indices back to **`merged_df`** rows, and returns a small table with **`semantic_score`** (inner product / cosine-style) plus titles, review text, and rating — parallel to `bm25_search`.
- **The loop over `queries`:** reuses the **same** query list as BM25; **`semantic_outputs`** stores one ranked `DataFrame` per query for comparison and discussion.


In [140]:
from src.semantic import load_semantic, semantic_search

semantic_model, semantic_index, _meta = load_semantic()

semantic_outputs = {}
for q in queries:
    semantic_outputs[q] = semantic_search(
        q, semantic_model, semantic_index, merged_df, top_k=5
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10133.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# ### Display semantic hits per query
# 
# The next cell displays the semantic search results for each query: for each query string, it prints a short header and **`display`s** the corresponding **`semantic_outputs[q]`** table, so you can inspect the top-5 matches found using semantic (dense vector) retrieval—not BM25!


In [141]:
for q in queries:
    
    print("=" * 50)
    print(f"Query: {q}")
    display(semantic_outputs[q])

Query: weight loss book


,product_title,review_title,review_text,semantic_score,rating
9979,Boost Your Metabolism Now:Choose The Right Foo...,Be Sure to Read ALL,I had planned to give this book one star. I ha...,0.545202,3.0
5797,Lose Weight Fast: Over 50 Incredible Weight Lo...,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,0.535709,4.0
6639,Dr. Corson's Top 5 Nutrition Tips: How To Lose...,Good Reminders for a Healthier Life,Fitness and Nutritional Consultant Dr. Candace...,0.535184,4.0
471,Why We Get Fat: And What to Do About It,Eye opening and presents a very strong case,Gary Taubes goes deep into his research and re...,0.477323,5.0
9235,The Poverty Cookbook: My Favorites,Love it !,I like the way the author writes. I love the t...,0.470126,5.0


Query: healthy eating guide


,product_title,review_title,review_text,semantic_score,rating
9704,Nutritional Healing: Special Nutrition with th...,What is good for you to help heal to body,This is a very informative book. I was chosen...,0.608272,5.0
6639,Dr. Corson's Top 5 Nutrition Tips: How To Lose...,Good Reminders for a Healthier Life,Fitness and Nutritional Consultant Dr. Candace...,0.542020,4.0
9979,Boost Your Metabolism Now:Choose The Right Foo...,Be Sure to Read ALL,I had planned to give this book one star. I ha...,0.541751,3.0
6628,Food Addiction Binge Eating And Hypoglycemia: ...,A Informative Book for Reactive Hypoglycemics,Tony Clearwater has written his book on binge ...,0.540425,4.0
1588,Fast And Easy Cabbage Recipes: An Guide To An ...,healthy eating,oh yea i forgot i had this book... i need to d...,0.531264,4.0


Query: beginner cookbook


...

,product_title,review_title,review_text,semantic_score,rating
1366,In Tandem (Equilibrium Book 2),Loved this,I loved how the author pens with her way of ar...,0.504657,5.0
213,Adulthood Is a Myth: A Sarah's Scribbles Colle...,Fun times,"I loved “reading” this book, finished in one s...",0.479337,5.0
529,Adulthood Is a Myth: A Sarah's Scribbles Colle...,This book is definitely reality!,I saw myself in it so many scenes. It is a mus...,0.478146,5.0
10446,Secret Burdens of the Heart: A Historical West...,Good book,I like it.,0.469584,4.0
485,Secret Burdens of the Heart: A Historical West...,she headed west to escape,Her father and 2nd wife were terrible. They we...,0.469584,5.0


Query: good books for beginners learning investing


,product_title,review_title,review_text,semantic_score,rating
9289,Investment: A History (Columbia Business Schoo...,Long and tiresome,The book is long and tiresome. I didn't learn ...,0.491082,3.0
1755,Market Wizards: Interviews with Top Traders,Great Book well worth the time and money to read!,I love the way Jack wrote about each one of th...,0.459878,5.0
8814,Little Kids Big Money: Tools for Teaching Kid ...,Must Read For Parents,This book really makes a lot of sense on how t...,0.444927,5.0
9402,The Node Beginner Book,A good resource for starting Node,"A good resource for starting Node, but it fall...",0.438229,3.0
4994,Free Trader on the High Seas (Free Trader Seri...,Another Gem,What a wonderful story! I just love the chara...,0.423957,5.0


Query: novels for long flights


,product_title,review_title,review_text,semantic_score,rating
10335,On Wings Of The Morning,great book,Really great read I suggest to every one I see...,0.514255,5.0
1819,Voyage (Paul's Travels),Still wish it would be completed,I originally stumbled upon the work in another...,0.512107,4.0
4571,A.L.I.V.E.: A Genetic Engineering Thriller (Th...,Fascinating but gross,The plot is dynamic and the book is a fascinat...,0.510917,4.0
4113,A.L.I.V.E.: A Genetic Engineering Thriller (Th...,Not so,Not so good,0.510917,4.0
4112,A.L.I.V.E.: A Genetic Engineering Thriller (Th...,Great perspective on Aliens living in U.S. Fac...,There have been many stories about the possibi...,0.510917,5.0
